<p style="text-align:center; font-size: 28px;">AI in Health Technologies (2026)</p>
<p style="text-align:center; font-size: 28px;">Project: Source or clinical content?</p>
<p style="text-align:center; font-size: 20px;">Alexandra Dahlberg, Heidi Tarkiainen, Joonatan Syrjänen</p>

### Table of Contents

* [Section 1: Introduction](#section_1)
* [Section 2: Setup](#section_2)
    * [Section 2.1: Packages](#section_2_1)
    * [Section 2.2: Settings](#section_2_2)
* [Section 3: Data](#section_3)
    * [Section 3.1: Loading the data](#section_3_1)
    * [Section 3.2: Initial look](#section_3_2)
    * [Section 3.3: Diagnosis codes](#section_3_3)
    * [Section 3.4: Note length](#section_3_4)
* [Section 4: Embeddings](#section_4)
    * [Section 4.1: Embedding model through Ollama](#section_4_1)
    * [Section 4.2: Embedding the notes](#section_4_2)
    * [Section 4.3: Table 1](#section_4_3)
    * [Section 4.4: FinBERT layers (only if FinBERT is available)](#section_4_4)
    * [Section 4.5: Check against Rami's textembed](#section_4_5)
* [Section 5: Projections](#section_5)
    * [Section 5.1: PCA](#section_5_1)
    * [Section 5.2: t-SNE](#section_5_2)
    * [Section 5.3: UMAP](#section_5_3)
* [Section 6: Separation in full and 2-D space](#section_6)
    * [Section 6.1: kNN label agreement](#section_6_1)
    * [Section 6.2: Silhouette](#section_6_2)
* [Section 7: Classifiers](#section_7)
    * [Section 7.1: Folds grouped by patient](#section_7_1)
    * [Section 7.2: Linear probe and MLP](#section_7_2)
    * [Section 7.3: Length and TF-IDF baselines](#section_7_3)
    * [Section 7.4: Chance level](#section_7_4)
* [Section 8: Synthetic data checks (Table 2)](#section_8)
    * [Section 8.1: Planted effects](#section_8_1)
    * [Section 8.2: Negative control](#section_8_2)
    * [Section 8.3: Shortcut test](#section_8_3)
* [Section 9: Q1 Source](#section_9)
    * [Section 9.1: UMAP grid (Fig. 2)](#section_9_1)
    * [Section 9.2: Separation and classifiers per label (Table 3)](#section_9_2)
* [Section 10: Q2 Diagnosis](#section_10)
    * [Section 10.1: Picture vs classifier per code (Fig. 3)](#section_10_1)
* [Section 11: Robustness and confounding](#section_11)
    * [Section 11.1: Seeds and UMAP settings](#section_11_1)
    * [Section 11.2: Confounding checks](#section_11_2)
    * [Section 11.3: Truncation check](#section_11_3)
* [Section 12: Choice of layer (Fig. 4)](#section_12)
* [Section 13: Clinical review](#section_13)
* [Section 14: Export](#section_14)

## 1. Introduction <a class="anchor" id="section_1"></a>

Something here.

## 2. Setup <a class="anchor" id="section_2"></a>

### 2.1. Packages <a class="anchor" id="section_2_1"></a>

In [ ]:
# List of those that could be relevant, needs to be checked and cleaned before submission

# Files and Ollama
import os
import requests

# Data and plots
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# PyTorch, for the classifiers and FinBERT (transformers is imported in 4.4, only if FinBERT is used)
import torch

# Projections
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap # pip install umap-learn

# kNN agreement and silhouette
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Classifiers, folds and baselines
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

### 2.2. Settings <a class="anchor" id="section_2_2"></a>

In [ ]:
# Synthetic or real data, everything below runs on both
data_source = "synthetic" # Change to "real" in Acamedic

if data_source == "synthetic":
    data_folder = "mock_data" # Mock tables in the repo
else:
    data_folder = "/path/to/acamedic/export" # Real export, same file names as mock_data

# Embedding model through Ollama, the main one is fixed before we look at the real data
model_name = "bge-m3" # "bge-m3" (main) or "qwen3" (comparison)

if model_name == "bge-m3":
    ollama_model = "bge-m3" # 567M parameters, 1024 numbers per note
else:
    ollama_model = "qwen3-embedding:8b" # 8B parameters, 4096 numbers per note

ollama_url = "http://localhost:11434" # Ollama on this computer, in Acamedic the address of the Azure ML compute
max_tokens = 8192 # Context length, set explicitly because Ollama otherwise cuts long notes without warning

# FinBERT for the layer check, only if it is available in Acamedic
run_finbert = False # True runs 4.4 and Section 12
finbert_path = "TurkuNLP/bert-base-finnish-cased-v1" # Local folder in Acamedic, no internet there
finbert_max_tokens = 512 # FinBERT can't read more than this

# GPU if there is one (Colab, Azure ML, Apple), otherwise CPU, only FinBERT and the classifiers use it
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps" # Apple GPU
else:
    device = "cpu"

# Seeds
seed = 42 # Life
np.random.seed(seed)
torch.manual_seed(seed)

# Analysis settings, fixed before we look at the real data
k = 15 # Neighbours for kNN agreement
projection_seeds = [0, 1, 2, 3, 4] # Seeds for t-SNE and UMAP, seed 0 is the main picture
umap_neighbours = [5, 15, 50] # Values of n_neighbors, 15 is the main picture
min_patients_per_code = 10 # Placeholder, smaller ICD codes are dropped
min_characters = 40 # Shorter notes are dropped, counted without the header line
sample_size = None # Random sample of notes for the real data, None keeps all
remove_header = True # True drops the first line (role, name, date), it gives away the hospital on its own. False is the sensitivity check

# Labels we colour the pictures by and measure separation for
labels_to_check = ["hospital", "unit", "specialty", "note_type", "writer_role", "year",
                   "length_quartile", "diagnosis_group", "diagnosis_code", "random_label"]

# Folders, and one set of files per model and header setting
results_folder = "results" # Figures, tables and run log go here
embeddings_folder = "embeddings" # Embeddings and labels for the rest of the team
os.makedirs(results_folder, exist_ok=True) # Makes the folder if it is missing
os.makedirs(embeddings_folder, exist_ok=True)

if remove_header:
    run_name = model_name + "_no_header" # E.g. bge-m3_no_header
else:
    run_name = model_name + "_with_header"

## 3. Data <a class="anchor" id="section_3"></a>

### 3.1. Loading the data <a class="anchor" id="section_3_1"></a>

In [ ]:
# Synthetic notes from the csv, one row per note (text + label columns)
# Like in Acamedic: needs the potilaskertomus table joined to the diagnosis table through the episode ID
# Print the number of notes and patients

# Notes and diagnoses, both tables use ; as the separator
notes = pd.read_csv(data_folder + "/Apotti_potilaskertomus.csv", sep=";")
diagnoses = pd.read_csv(data_folder + "/Apotti_diagnoosi.csv", sep=";")

# Main diagnosis per care episode (shjakso), one row each
main_diagnoses = diagnoses[diagnoses["paadiagnoosi"] == True] # Main diagnosis rows only
main_diagnoses = main_diagnoses.drop_duplicates(subset="shjakso_id") # A few episodes have the same code twice
main_diagnoses = main_diagnoses[["shjakso_id", "diagnoosi_koodi", "potilas_ika_vuosina", "potilas_sukupuoli_selite"]]

# Diagnosis, age and sex onto every note through the care episode
notes = notes.merge(main_diagnoses, on="shjakso_id", how="left")

# Labels with English names, so the rest of the notebook is easier to read
notes["note_id"] = notes["merkinta_id"] # Links the embeddings back to the notes
notes["patient_id"] = notes["henkilotunnus"] # Pseudonym, used to group the folds
notes["hospital"] = notes["toimipiste_selite"].str.split(", ").str[-1] # E.g. Tornisairaala, after the comma
notes["unit"] = notes["toimipiste_selite"] # Ward, outpatient clinic or theatre, with the hospital
notes["specialty"] = notes["toimipiste_eala_selite"] # Verisuonikirurgia or Anestesiologia
notes["note_type"] = notes["merkinta_valilehti_selite"] # E.g. Päivittäismerkintä, Leikkauskertomus
notes["year"] = pd.to_datetime(notes["merkinta_luonti_hetki"]).dt.year # Year the note was written
notes["diagnosis_code"] = notes["diagnoosi_koodi"] # E.g. I70.2
notes["diagnosis_group"] = notes["diagnoosi_koodi"].str[:3] # E.g. I70
notes["age"] = notes["potilas_ika_vuosina"]
notes["sex"] = notes["potilas_sukupuoli_selite"]
notes["text"] = notes["merkinta_teksti"]

# Header line, e.g. "Erikoislääkäri KARJALAINEN, ANNA 18.05.2025 08:58"
notes["header"] = notes["text"].str.split("\n").str[0] # First line
notes["body"] = notes["text"].str.split("\n", n=1).str[1] # Everything after the first line
notes["body"] = notes["body"].fillna("") # Notes with only a header line become empty
role_and_surname = notes["header"].str.split(", ").str[0] # E.g. "Erikoislääkäri KARJALAINEN"
notes["writer_role"] = role_and_surname.str.rsplit(" ", n=1).str[0] # Drops the surname, e.g. "Erikoislääkäri"
notes["physician"] = notes["writer_role"].isin(["Erikoislääkäri", "Erikoistuva lääkäri"]) # Check the real role names in 3.2

# Drop the header line if the setting says so
if remove_header:
    notes["text"] = notes["body"]

print("Notes: %d"
      % notes.shape[0])
print("Patients: %d"
      % notes["patient_id"].nunique())
print("Care episodes: %d"
      % notes["shjakso_id"].nunique())
print("Notes without a main diagnosis: %d"
      % notes["diagnosis_code"].isnull().sum())

### 3.2. Initial look <a class="anchor" id="section_3_2"></a>

In [ ]:
# Very short notes, counted without the header so both header settings keep the same notes
notes["characters"] = notes["body"].str.len() # Length in characters
short = notes["characters"] < min_characters # E.g. "Uusi aika annettu."
print("Notes under %d characters dropped: %d (%.1f %%)"
      % (min_characters, short.sum(), 100 * short.mean()))
notes = notes[~short] # ~ means not

# Random sample for the real data, all notes for the synthetic data
if sample_size is not None:
    notes = notes.sample(n=sample_size, random_state=seed)
    notes = notes.sort_index() # Back to the original order
notes = notes.reset_index(drop=True) # Row n of notes is row n of the embeddings

# Negative control, a random label per patient that should not separate at all
patients = notes["patient_id"].unique() # One entry per patient
random_generator = np.random.default_rng(seed)
random_groups = random_generator.choice(["A", "B"], size=len(patients)) # Coin flip per patient
random_labels = pd.DataFrame({"patient_id": patients, "random_label": random_groups})
notes = notes.merge(random_labels, on="patient_id", how="left")

print("Notes left: %d, patients: %d"
      % (notes.shape[0], notes["patient_id"].nunique()))

# Notes per label
for label in ["hospital", "unit", "specialty", "note_type", "writer_role", "year"]:
    print()
    print(notes[label].value_counts())

# Notes per patient, a few long ward stays give many notes
notes_per_patient = notes.groupby("patient_id").size()
print()
print("Notes per patient: median %d, largest %d"
      % (notes_per_patient.median(), notes_per_patient.max()))

### 3.3. Diagnosis codes <a class="anchor" id="section_3_3"></a>

In [ ]:
# Main ICD-10 code per care episode, at the finer level (e.g. I70.2, I71.4, I65.2)
# A patient can have more than one code across care episodes

# Patients per code
patients_per_code = notes.groupby("diagnosis_code")["patient_id"].nunique()
print(patients_per_code.sort_values(ascending=False))

# Rare codes are set to missing, the notes stay in for the source labels
kept_codes = patients_per_code[patients_per_code >= min_patients_per_code].index # Codes with enough patients
notes["diagnosis_code"] = notes["diagnosis_code"].where(notes["diagnosis_code"].isin(kept_codes)) # Missing for the rest

# Same for the three-character groups
patients_per_group = notes.groupby("diagnosis_group")["patient_id"].nunique()
kept_groups = patients_per_group[patients_per_group >= min_patients_per_code].index
notes["diagnosis_group"] = notes["diagnosis_group"].where(notes["diagnosis_group"].isin(kept_groups))

print()
print("Codes kept: %d of %d, notes with a kept code: %d"
      % (len(kept_codes), len(patients_per_code), notes["diagnosis_code"].notnull().sum()))
print("Groups kept: %d of %d"
      % (len(kept_groups), len(patients_per_group)))

### 3.4. Note length <a class="anchor" id="section_3_4"></a>

In [ ]:
# Length of the note body in characters and words, tokens come from the embedding model in 4.2
notes["words"] = notes["body"].str.split().str.len() # Words between spaces and line breaks

# Length as a label too, so we can see if the pictures just sort short from long notes
length_quartile = pd.qcut(notes["characters"], 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"]) # Four equal-sized groups
notes["length_quartile"] = length_quartile.astype(str)

print("Characters per note: median %d, largest %d"
      % (notes["characters"].median(), notes["characters"].max()))
print("Words per note: median %d, largest %d"
      % (notes["words"].median(), notes["words"].max()))

## 4. Embeddings <a class="anchor" id="section_4"></a>

### 4.1. Embedding model through Ollama <a class="anchor" id="section_4_1"></a>

The model runs in Ollama, on our own computers and in Acamedic on Azure ML, so the same code works in both places. Before the first run, install Ollama and pull the model in a terminal: `ollama pull bge-m3` (and `ollama pull qwen3-embedding:8b` for the comparison). Ollama gives one pooled vector per note, normalised to length 1.

In [ ]:
# One note in, one vector out
# options num_ctx sets the context length, otherwise Ollama uses its own shorter default and cuts long notes

def embed(text): # One note through Ollama, gives its vector and the number of tokens read
    response = requests.post(ollama_url + "/api/embed", json={
        "model": ollama_model,
        "input": text,
        "options": {"num_ctx": max_tokens},
    })
    if response.status_code != 200:
        raise RuntimeError("Ollama error: " + response.text) # E.g. model not pulled, or NaN from a long note
    answer = response.json()
    vector = answer["embeddings"][0] # One note in, so one vector out
    tokens = answer["prompt_eval_count"] # Tokens Ollama read, stops at the context length
    return vector, tokens

### 4.2. Embedding the notes <a class="anchor" id="section_4_2"></a>

Runs once per model and header setting and saves the result. On later runs the saved embeddings are loaded, so this part needs Ollama only the first time.

In [ ]:
# Every note through the model one at a time, saved so this runs only once
embeddings_file = embeddings_folder + "/" + run_name + ".npy" # E.g. embeddings/bge-m3_no_header.npy
labels_file = embeddings_folder + "/" + run_name + "_labels.csv" # Note IDs, labels and tokens, same row order, no text
label_columns = ["note_id", "patient_id", "shjakso_id", "physician", "age", "sex",
                 "characters", "words", "tokens", "cut"] + labels_to_check

if os.path.exists(embeddings_file):
    # Saved earlier, loaded instead of embedding again
    embeddings = np.load(embeddings_file)
    saved_labels = pd.read_csv(labels_file)
    if saved_labels["note_id"].tolist() != notes["note_id"].tolist():
        raise RuntimeError("The saved embeddings are for other notes, delete " + embeddings_file + " and embed again")
    notes["tokens"] = saved_labels["tokens"]
    notes["cut"] = saved_labels["cut"]
    print("Loaded %d embeddings from %s"
          % (embeddings.shape[0], embeddings_file))
else:
    # Context check: a text far over max_tokens is cut, and the tokens read show where the limit is
    long_text = "Potilas voi hyvin. " * 5000 # About 25 000 tokens
    test_vector, token_limit = embed(long_text)
    print("Tokens read from a very long text: %d (max_tokens is %d)"
          % (token_limit, max_tokens)) # Clearly lower means num_ctx was ignored

    # All notes
    N = notes.shape[0]
    embeddings = np.zeros((N, len(test_vector)), dtype=np.float32) # Shape (N, 1024) for BGE-M3
    tokens = np.zeros(N, dtype=int) # Tokens read per note
    for n in range(N):
        vector, note_tokens = embed(notes["text"][n])
        embeddings[n] = vector
        tokens[n] = note_tokens
        if n % 1000 == 0:
            print("Embedded %d of %d notes"
                  % (n, N))
    notes["tokens"] = tokens
    notes["cut"] = tokens >= token_limit # Notes that reached the limit were cut

    # Saved for the rest of the team, labels without any text
    np.save(embeddings_file, embeddings)
    notes[label_columns].to_csv(labels_file, index=False)
    print("Saved %s and %s"
          % (embeddings_file, labels_file))

print("Embeddings, shape (%d, %d)"
      % embeddings.shape)

### 4.3. Table 1 <a class="anchor" id="section_4_3"></a>

In [ ]:
# Column of Table 1 for this data, tokens counted by the embedding model's own tokenizer
table1 = pd.DataFrame({
    "item": ["Notes", "Patients", "ICD-10 codes", "Median tokens per note", "Notes over 512 tokens (%)"],
    data_source: [notes.shape[0],
                  notes["patient_id"].nunique(),
                  notes["diagnosis_code"].nunique(),
                  notes["tokens"].median(),
                  100 * np.mean(notes["tokens"] > 512)],
})
table1.to_csv(results_folder + "/" + run_name + "_table1_" + data_source + ".csv", index=False)
table1.round(1)

### 4.4. FinBERT layers (only if FinBERT is available) <a class="anchor" id="section_4_4"></a>

Ollama only gives the final pooled vector, so the layer check needs FinBERT. Runs only when `run_finbert = True`.

In [ ]:
# e = 1/T * SUM over t of h_t, the mean of the T token vectors, for every layer
# One note at a time, so there is no padding and the plain mean is the masked mean

if run_finbert:
    from transformers import AutoTokenizer, AutoModel # Only needed here

    tokenizer = AutoTokenizer.from_pretrained(finbert_path)
    finbert = AutoModel.from_pretrained(finbert_path, output_hidden_states=True) # Gives every layer, not just the last
    finbert = finbert.to(device)
    finbert.eval() # Inference mode, the weights stay frozen

    N = notes.shape[0]
    number_of_layers = finbert.config.num_hidden_layers + 1 # 13 for FinBERT, the input layer and 12 layers
    hidden_size = finbert.config.hidden_size # 768 for FinBERT
    layer_embeddings = np.zeros((number_of_layers, N, hidden_size), dtype=np.float32) # Shape (13, N, 768)
    finbert_tokens = np.zeros(N, dtype=int) # Tokens per note before cutting, for the share over 512

    for n in range(N):
        all_tokens = tokenizer(notes["text"][n])["input_ids"] # Whole note, only to count the tokens
        finbert_tokens[n] = len(all_tokens)
        encoded = tokenizer(notes["text"][n], truncation=True, max_length=finbert_max_tokens, return_tensors="pt") # First 512 tokens
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        with torch.no_grad(): # No gradients needed, faster
            output = finbert(input_ids=input_ids, attention_mask=attention_mask)
        for layer in range(number_of_layers):
            token_vectors = output.hidden_states[layer][0] # One vector per token, shape (T, 768)
            layer_embeddings[layer, n] = token_vectors.mean(dim=0).cpu().numpy() # Mean pooling
        if n % 1000 == 0:
            print("FinBERT: %d of %d notes"
                  % (n, N))

    print("FinBERT notes over %d tokens: %.1f %%"
          % (finbert_max_tokens, 100 * np.mean(finbert_tokens > finbert_max_tokens)))

### 4.5. Check against Rami's textembed <a class="anchor" id="section_4_5"></a>

In [ ]:
# Run Rami's textembed on the same notes
# Our vectors should match his, if not something is off in the pooling

## 5. Projections <a class="anchor" id="section_5"></a>

In [ ]:
# Helper for the projection plots, one panel per label, each class in its own colour, notes without a label in grey
# The notes are drawn in a random order, otherwise the class drawn last covers the others

projections = {} # 2-D coordinates of every run, shape (N, 2) each, filled in 5.1-5.3
drawing_order = np.random.default_rng(seed).permutation(notes.shape[0]) # Random order of the notes, same in every plot

def plot_by_labels(points, title, file_name): # Scatter of the 2-D points, coloured by every label in labels_to_check
    number_of_rows = int(np.ceil(len(labels_to_check) / 5)) # Five panels per row
    fig, axes = plt.subplots(number_of_rows, 5, figsize=(25, 5 * number_of_rows))
    for index in range(len(labels_to_check)):
        label = labels_to_check[index]
        ax = axes.flat[index] # Panels in reading order
        classes = sorted(notes[label].dropna().unique()) # Classes of this label
        colours = plt.cm.tab20(np.linspace(0, 1, len(classes))) # Up to 20 different colours, RGBA
        point_colours = np.zeros((notes.shape[0], 4)) # One RGBA colour per note
        point_colours[:] = [0.85, 0.85, 0.85, 1] # Light grey for notes without a label
        for class_index in range(len(classes)):
            in_class = (notes[label] == classes[class_index]).to_numpy() # Notes of this class
            point_colours[in_class] = colours[class_index]
            ax.scatter([], [], color=colours[class_index], label=classes[class_index]) # Empty, only for the legend
        ax.scatter(points[drawing_order, 0], points[drawing_order, 1], s=1, c=point_colours[drawing_order])
        ax.set_title(label)
        ax.legend(fontsize="x-small")
    fig.suptitle(title)
    fig.savefig(results_folder + "/" + run_name + "_" + file_name + ".png")
    plt.show()

### 5.1. PCA <a class="anchor" id="section_5_1"></a>

In [ ]:
# PCA to 2-D, the simple linear projection, mostly as a sanity check
pca = PCA(n_components=2)
projections["pca"] = pca.fit_transform(embeddings) # Shape (N, 2)

print("Variance explained by the two components: %.1f %%"
      % (100 * np.sum(pca.explained_variance_ratio_)))
plot_by_labels(projections["pca"], "PCA", "pca")

### 5.2. t-SNE <a class="anchor" id="section_5_2"></a>

In [ ]:
# t-SNE to 2-D, cosine, perplexity 30, PCA start, one run per seed
for projection_seed in projection_seeds:
    tsne = TSNE(metric="cosine", perplexity=30, init="pca", random_state=projection_seed)
    projections["tsne_seed%d" % projection_seed] = tsne.fit_transform(embeddings) # Shape (N, 2)
    print("t-SNE seed %d done"
          % projection_seed)

plot_by_labels(projections["tsne_seed0"], "t-SNE, seed 0", "tsne")

### 5.3. UMAP <a class="anchor" id="section_5_3"></a>

In [ ]:
# UMAP to 2-D, cosine, min_dist 0.1, every n_neighbors with every seed
for n_neighbors in umap_neighbours:
    for projection_seed in projection_seeds:
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, metric="cosine", random_state=projection_seed, n_jobs=1) # A seed means one core anyway, n_jobs=1 stops the warning
        projections["umap_n%d_seed%d" % (n_neighbors, projection_seed)] = reducer.fit_transform(embeddings) # Shape (N, 2)
    print("UMAP n_neighbors %d done"
          % n_neighbors)

plot_by_labels(projections["umap_n15_seed0"], "UMAP, n_neighbors 15, seed 0", "umap")

# All coordinates in one table with the note IDs, these may leave Acamedic (no text)
coordinates = pd.DataFrame({"note_id": notes["note_id"]})
for name in projections:
    coordinates[name + "_x"] = projections[name][:, 0]
    coordinates[name + "_y"] = projections[name][:, 1]
coordinates.to_csv(results_folder + "/" + run_name + "_coordinates.csv", index=False)

## 6. Separation in full and 2-D space <a class="anchor" id="section_6"></a>

### 6.1. kNN label agreement <a class="anchor" id="section_6_1"></a>

For every note we look at its 15 nearest notes and count how many have the same label. Notes from the same patient are skipped, because copy-forward makes a patient's own notes each other's nearest neighbours and would make every patient-level label (diagnosis, hospital) look separated. The share is compared with what chance would give, so a lift of 1 means no better than chance and 2 means twice as many same-label neighbours as chance.

In [ ]:
# observed(n) = share of the k nearest notes of note n with the same label, notes of the same patient skipped
# chance(n) = share of the label of note n among all notes of the other patients
# lift = mean of observed / mean of chance

def knn_lift(points, labels, patients, metric): # kNN label agreement as lift, overall and per class
    # Only notes with a label, e.g. rare codes are left out
    has_label = labels.notnull().to_numpy()
    points = points[has_label]
    labels = labels.to_numpy()[has_label]
    patients = patients.to_numpy()[has_label]
    N = len(labels)

    # Nearest neighbours, extra ones so that k are left after skipping the same patient
    notes_per_patient = pd.Series(patients).value_counts()
    number_of_neighbours = min(N, k + notes_per_patient.max()) # Never more than there are notes
    nearest = NearestNeighbors(n_neighbors=number_of_neighbours, metric=metric)
    nearest.fit(points)
    _, neighbours = nearest.kneighbors(points) # The _ is the distances, not needed, shape (N, number_of_neighbours)

    # Observed share for every note
    observed = np.zeros(N)
    for n in range(N):
        other_patients = neighbours[n][patients[neighbours[n]] != patients[n]] # Skips the note itself and the same patient
        nearest_k = other_patients[:k] # The k nearest from other patients
        observed[n] = np.mean(labels[nearest_k] == labels[n]) # Share with the same label

    # Chance level for every note
    table = pd.DataFrame({"label": labels, "patient": patients})
    label_count = table.groupby("label")["label"].transform("size") # Notes with this label
    same_patient_count = table.groupby(["patient", "label"])["label"].transform("size") # Of those, from the same patient
    patient_count = table.groupby("patient")["patient"].transform("size") # Notes of this patient
    chance = (label_count - same_patient_count) / (N - patient_count) # Share of the label among the other patients' notes
    chance = chance.to_numpy()

    # Lift overall and for each class
    overall = np.mean(observed) / np.mean(chance)
    classes = np.unique(labels)
    class_lifts = np.zeros(len(classes))
    for index in range(len(classes)):
        in_class = labels == classes[index] # Notes of this class
        if np.mean(chance[in_class]) > 0:
            class_lifts[index] = np.mean(observed[in_class]) / np.mean(chance[in_class])
        else:
            class_lifts[index] = np.nan # Only one patient has this class, nothing to compare with
    per_class = pd.Series(class_lifts, index=classes)
    return overall, per_class


# Lift for every label, cosine in the full space and Euclidean in the three main pictures
rows = [] # One row per label
for label in labels_to_check:
    lift_full, _ = knn_lift(embeddings, notes[label], notes["patient_id"], "cosine") # The _ is the per-class lifts, not needed here
    lift_pca, _ = knn_lift(projections["pca"], notes[label], notes["patient_id"], "euclidean")
    lift_tsne, _ = knn_lift(projections["tsne_seed0"], notes[label], notes["patient_id"], "euclidean")
    lift_umap, _ = knn_lift(projections["umap_n15_seed0"], notes[label], notes["patient_id"], "euclidean")
    rows.append([label, lift_full, lift_pca, lift_tsne, lift_umap])

# Diagnosis again with physicians' notes only, nursing and physiotherapy notes rarely name the diagnosis
physician = notes["physician"].to_numpy()
for label in ["diagnosis_group", "diagnosis_code"]:
    lift_full, _ = knn_lift(embeddings[physician], notes[label][physician], notes["patient_id"][physician], "cosine")
    lift_pca, _ = knn_lift(projections["pca"][physician], notes[label][physician], notes["patient_id"][physician], "euclidean")
    lift_tsne, _ = knn_lift(projections["tsne_seed0"][physician], notes[label][physician], notes["patient_id"][physician], "euclidean")
    lift_umap, _ = knn_lift(projections["umap_n15_seed0"][physician], notes[label][physician], notes["patient_id"][physician], "euclidean")
    rows.append([label + " (physicians)", lift_full, lift_pca, lift_tsne, lift_umap])

knn_table = pd.DataFrame(rows, columns=["label", "full", "pca", "tsne", "umap"])
knn_table["full_minus_umap"] = knn_table["full"] - knn_table["umap"] # How much the main picture distorts
knn_table.to_csv(results_folder + "/" + run_name + "_knn_lift.csv", index=False)
knn_table.round(2)

In [ ]:
# Lift per ICD code, the x axis of Fig. 3, the classifier AUROC per code comes from Section 7
_, codes_full = knn_lift(embeddings, notes["diagnosis_code"], notes["patient_id"], "cosine")
_, codes_umap = knn_lift(projections["umap_n15_seed0"], notes["diagnosis_code"], notes["patient_id"], "euclidean")
_, codes_full_physicians = knn_lift(embeddings[physician], notes["diagnosis_code"][physician], notes["patient_id"][physician], "cosine")
_, codes_umap_physicians = knn_lift(projections["umap_n15_seed0"][physician], notes["diagnosis_code"][physician], notes["patient_id"][physician], "euclidean")

code_table = pd.DataFrame({
    "patients": notes.groupby("diagnosis_code")["patient_id"].nunique(), # Dot size in Fig. 3
    "notes": notes.groupby("diagnosis_code").size(),
    "lift_full": codes_full,
    "lift_umap": codes_umap,
    "lift_full_physicians": codes_full_physicians,
    "lift_umap_physicians": codes_umap_physicians,
})
code_table.to_csv(results_folder + "/" + run_name + "_knn_lift_per_code.csv", index_label="diagnosis_code")
code_table.round(2)

### 6.2. Silhouette <a class="anchor" id="section_6_2"></a>

In [ ]:
# Silhouette per label, secondary measure from -1 to 1, around 0 means the classes overlap
# Unlike the kNN lift it does not skip notes of the same patient
rows = [] # One row per label
for label in labels_to_check:
    has_label = notes[label].notnull().to_numpy() # Notes with a label
    labels = notes[label][has_label]
    silhouette_full = silhouette_score(embeddings[has_label], labels, metric="cosine")
    silhouette_pca = silhouette_score(projections["pca"][has_label], labels)
    silhouette_tsne = silhouette_score(projections["tsne_seed0"][has_label], labels)
    silhouette_umap = silhouette_score(projections["umap_n15_seed0"][has_label], labels)
    rows.append([label, silhouette_full, silhouette_pca, silhouette_tsne, silhouette_umap])

silhouette_table = pd.DataFrame(rows, columns=["label", "full", "pca", "tsne", "umap"])
silhouette_table.to_csv(results_folder + "/" + run_name + "_silhouette.csv", index=False)
silhouette_table.round(2)

## 7. Classifiers <a class="anchor" id="section_7"></a>

### 7.1. Folds grouped by patient <a class="anchor" id="section_7_1"></a>

In [22]:
# 5 folds grouped by patient (GroupKFold), so the same patient is never in both train and test
# Small validation split inside the training folds for early stopping

### 7.2. Linear probe and MLP <a class="anchor" id="section_7_2"></a>

In [23]:
# Linear probe: one nn.Linear on the frozen embeddings, cross entropy, Adam
# MLP: one hidden layer, otherwise the same
# Same training loop for both, early stopping on the validation loss
# Macro AUROC and balanced accuracy per label, plus AUROC per class for Q2

### 7.3. Length and TF-IDF baselines <a class="anchor" id="section_7_3"></a>

In [24]:
# Same linear model, but trained on note length only (characters, words, tokens)
# And on TF-IDF features, fitted inside each training fold so nothing leaks from the test fold
# If these do as well as the embedding, the embedding adds little

### 7.4. Chance level <a class="anchor" id="section_7_4"></a>

In [25]:
# Shuffle the labels and rerun, gives the chance level for each label

## 8. Synthetic data checks (Table 2) <a class="anchor" id="section_8"></a>

Only for the synthetic data, since we know what we planted there.

### 8.1. Planted effects <a class="anchor" id="section_8_1"></a>

In [26]:
# Does the pipeline find the planted source and diagnosis effects?
# Diagnosis effects should come out in the order we planted them, weak to strong

### 8.2. Negative control <a class="anchor" id="section_8_2"></a>

In [27]:
# The label with no signal, we expect kNN lift and AUROC around chance

### 8.3. Shortcut test <a class="anchor" id="section_8_3"></a>

In [28]:
# Train the diagnosis classifier where diagnosis is linked to unit style, test where the link is broken
# Compare AUROC with and without the link, and check if the UMAP showed the unit clusters beforehand
# Table 2 is put together from 8.1-8.3 and saved as table2.csv

## 9. Q1 Source <a class="anchor" id="section_9"></a>

### 9.1. UMAP grid (Fig. 2) <a class="anchor" id="section_9_1"></a>

In [ ]:
# Same UMAP four times, coloured by note type, unit, year and note length (Fig. 2)
umap_points = projections["umap_n15_seed0"][drawing_order] # Main picture, notes in the random drawing order from Section 5
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Note type
note_type = notes["note_type"][drawing_order] # Same order as the points
note_types = sorted(note_type.unique())
colours = plt.cm.tab10(np.linspace(0, 1, len(note_types))) # One colour per note type, RGBA
point_colours = np.zeros((notes.shape[0], 4)) # One RGBA colour per note
for index in range(len(note_types)):
    in_class = (note_type == note_types[index]).to_numpy() # Notes of this type
    point_colours[in_class] = colours[index]
    axes[0, 0].scatter([], [], color=colours[index], label=note_types[index]) # Empty, only for the legend
axes[0, 0].scatter(umap_points[:, 0], umap_points[:, 1], s=1, c=point_colours)
axes[0, 0].set_title("Note type")
axes[0, 0].legend(fontsize="x-small")

# Unit
unit = notes["unit"][drawing_order]
units = sorted(unit.unique())
colours = plt.cm.tab10(np.linspace(0, 1, len(units)))
point_colours = np.zeros((notes.shape[0], 4))
for index in range(len(units)):
    in_class = (unit == units[index]).to_numpy() # Notes of this unit
    point_colours[in_class] = colours[index]
    axes[0, 1].scatter([], [], color=colours[index], label=units[index])
axes[0, 1].scatter(umap_points[:, 0], umap_points[:, 1], s=1, c=point_colours)
axes[0, 1].set_title("Unit")
axes[0, 1].legend(fontsize="x-small")

# Year, a colour scale because the years are in order
year_points = axes[1, 0].scatter(umap_points[:, 0], umap_points[:, 1], s=1, c=notes["year"][drawing_order])
fig.colorbar(year_points, ax=axes[1, 0])
axes[1, 0].set_title("Year")

# Note length, log scale so the few very long notes don't take the whole colour range
length_points = axes[1, 1].scatter(umap_points[:, 0], umap_points[:, 1], s=1, c=np.log10(notes["characters"][drawing_order]))
fig.colorbar(length_points, ax=axes[1, 1], label="log10 of characters")
axes[1, 1].set_title("Note length")

# UMAP axes have no units
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(results_folder + "/" + run_name + "_fig2_umap_grid.png", dpi=300)
plt.show()

### 9.2. Separation and classifiers per label (Table 3) <a class="anchor" id="section_9_2"></a>

In [30]:
# One row per label: kNN lift full and 2-D, AUROC for embedding, TF-IDF, length and chance
# Save as table3.csv

## 10. Q2 Diagnosis <a class="anchor" id="section_10"></a>

### 10.1. Picture vs classifier per code (Fig. 3) <a class="anchor" id="section_10_1"></a>

In [31]:
# Per ICD code: 2-D kNN lift (x) against classifier AUROC (y), one dot per code
# Dot size by number of patients
# Spearman with a bootstrap CI only if there are 10+ codes, otherwise we just describe it
# Save as fig3_picture_vs_classifier.png

## 11. Robustness and confounding <a class="anchor" id="section_11"></a>

### 11.1. Seeds and UMAP settings <a class="anchor" id="section_11_1"></a>

In [ ]:
# 2-D kNN lift for every t-SNE seed and every UMAP setting and seed
# The spread shows how much one picture can be trusted, the full-space value is in 6.1
rows = [] # One row per label and run
for label in labels_to_check:
    for projection_seed in projection_seeds:
        lift, _ = knn_lift(projections["tsne_seed%d" % projection_seed], notes[label], notes["patient_id"], "euclidean")
        rows.append([label, "t-SNE", projection_seed, lift])
    for n_neighbors in umap_neighbours:
        for projection_seed in projection_seeds:
            lift, _ = knn_lift(projections["umap_n%d_seed%d" % (n_neighbors, projection_seed)], notes[label], notes["patient_id"], "euclidean")
            rows.append([label, "UMAP n_neighbors %d" % n_neighbors, projection_seed, lift])

robustness = pd.DataFrame(rows, columns=["label", "projection", "seed", "lift"])
robustness.to_csv(results_folder + "/" + run_name + "_knn_lift_seeds.csv", index=False)

# Mean, smallest and largest lift over the five seeds
robustness_summary = robustness.groupby(["label", "projection"])["lift"].agg(["mean", "min", "max"])
robustness_summary.round(2)

### 11.2. Confounding checks <a class="anchor" id="section_11_2"></a>

In [33]:
# Source labels within the same ICD code, diagnosis within the same unit and note type
# Same kNN lift and classifiers as above, just on the subsets
# Optional: train diagnosis on one note type, test on another

### 11.3. Truncation check <a class="anchor" id="section_11_3"></a>

In [ ]:
# Notes the model cut at the context length, counted in 4.2
print("Notes cut at %d tokens: %d of %d (%.2f %%)"
      % (max_tokens, notes["cut"].sum(), notes.shape[0], 100 * notes["cut"].mean()))
print("Longest note: %d tokens"
      % notes["tokens"].max())

## 12. Choice of layer (Fig. 4) <a class="anchor" id="section_12"></a>

In [ ]:
# kNN lift in the full space for every FinBERT layer, one line per label (Fig. 4)
if run_finbert:
    layer_lifts = np.zeros((len(labels_to_check), number_of_layers)) # One row per label, one column per layer
    for label_index in range(len(labels_to_check)):
        label = labels_to_check[label_index]
        for layer in range(number_of_layers):
            lift, _ = knn_lift(layer_embeddings[layer], notes[label], notes["patient_id"], "cosine")
            layer_lifts[label_index, layer] = lift
        print("%s done"
              % label)

    # Saved as a table, one row per label
    layer_table = pd.DataFrame(layer_lifts, index=labels_to_check)
    layer_table.to_csv(results_folder + "/" + run_name + "_finbert_layers.csv", index_label="label")

    # Plotting
    fig, ax = plt.subplots()
    for label_index in range(len(labels_to_check)):
        ax.plot(range(number_of_layers), layer_lifts[label_index], label=labels_to_check[label_index])
    ax.axvline(number_of_layers - 1, linestyle="--", color="grey") # Last layer, the one embedding models use
    ax.set_title("kNN lift across FinBERT layers")
    ax.set_xticks(range(number_of_layers)) # Whole layers only
    ax.set_xlabel("Layer")
    ax.set_ylabel("kNN lift, full space")
    ax.legend(fontsize="x-small")
    fig.savefig(results_folder + "/" + run_name + "_fig4_layers.png", dpi=300)
    plt.show()

## 13. Clinical review <a class="anchor" id="section_13"></a>

Done inside Acamedic only.

In [36]:
# Pick a handful of notes from each visible UMAP cluster (note IDs only)
# Read them inside Acamedic and write down what separates the clusters: template, structure or content
# Nothing from here leaves Acamedic except those categories

## 14. Export <a class="anchor" id="section_14"></a>

In [37]:
# Save all figures and tables to results/ with the run date
# Quick check that no text column ends up in anything we export
# Write the run log (settings, numbers of notes, package versions) for the README